# Khung 1 evaluation — RAG vs SAG (retrieval + answer)

## 1. Introduction

This notebook evaluates the **SAG Vietnam Legal** pipeline on the **Khung 1** finance-oriented statute pack:

```text
Query → Hybrid (BM25 + dense) → Voyage rerank  →  [RAG evidence]
                              ↘ SAG expand     →  [SAG evidence]
                              → LLM draft (optional)
```

**Why evaluate in layers?**  
SAG’s job is to improve the *evidence pack* (e.g. recover parent Điều/Khoản, bridge related laws).  
A fluent LLM answer can hide retrieval failure — and a good retrieval pack can still be wasted if the draft ignores repairs.  
So we score **retrieval first**, then optionally **answer phrase hits**.

**Research question (project):**  
*Does hybrid + Voyage + SAG retrieve more of the legally necessary provisions than hybrid + Voyage alone (RAG), on Vietnamese Khung 1 questions?*

Gold labels live in `evaluation/datasets/khung1_gold_v0.jsonl` (hand-authored, small v0 set — expand later).

## 2. Evaluation criteria

| Layer | Criterion | Pass idea |
|-------|-----------|-----------|
| **A. Document routing** | Needed `document_id`(s) appear in evidence | Doc recall |
| **B. Provision coverage** | Needed Điều / Khoản / Điểm appear | Provision recall |
| **C. Evidence noise** | Share of retrieved chunks that touch gold provisions/docs | Precision@K (bag) |
| **D. Answer groundedness (light)** | Required legal phrases appear in the draft | Phrase recall |
| **E. Abstention quality** | Model abstains only when gold evidence is missing | Manual / flag |

### What we are *not* claiming in v0
- Full legal correctness F1 against a lawyer-authored essay answer  
- Hindsight verification catch-rate (not wired yet)  
- Contract-review issue detection  

### Arms compared
- **RAG** = Voyage shortlist only (`use_sag=False`)  
- **SAG** = same seeds + structural/semantic expansion  

Optional: `SKIP_VOYAGE=True` uses hybrid top‑k as seeds (avoids Voyage RPM limits).

## 3. Scoring definitions

For each gold item with must-docs $D^*$ and must-provisions $P^*$, and retrieved pack $E$ of size $K$:

$$
\mathrm{DocRecall} = \frac{|\{d \in D^* : d \text{ appears in } E\}|}{|D^*|}
$$

$$
\mathrm{ProvRecall} = \frac{|\{p \in P^* : p \text{ matched by some chunk in } E\}|}{|P^*|}
$$

A provision matches if `document_id` + `article` match, and `clause` / `point` match when specified in gold.

$$
\mathrm{Precision@K} = \frac{|\{c \in E : c \text{ touches a gold doc/provision}\}|}{K}
$$

$$
\mathrm{F1}_{prov} = F1(\mathrm{Precision@K}, \mathrm{ProvRecall})
$$

**Primary leaderboard metric for SAG vs RAG:** mean **ProvRecall** (then DocRecall).  
Precision is secondary — SAG is allowed to add context and may lower Precision@K.

## 4. Setup

Run from the **repo root** (or ensure it is on `sys.path`).  
Needs: corpus JSON, embedding cache (built on first run), `.env` with keys if using Voyage/Qwen.

In [1]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").is_file():
    # Allow opening the notebook from notebooks/
    if (ROOT.parent / "pyproject.toml").is_file():
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

from evaluation.metrics import f1, load_gold, mean, score_answer, score_retrieval
from sag_legal.generation import generate_draft
from sag_legal.ingestion import KHUNG1_DOC_IDS, flatten_chunks, ingest_corpus
from sag_legal.retrieval.embeddings import embed_chunks
from sag_legal.retrieval.hybrid import search_hybrid
from sag_legal.reranking import rerank
from sag_legal.sag import build_index, expand
from sag_legal.settings import get_settings

GOLD_PATH = ROOT / "evaluation" / "datasets" / "khung1_gold_v0.jsonl"
RAW_JSON = ROOT / "data" / "raw" / "uts_vlc_processed.json"
CACHE_NPZ = ROOT / "data" / "processed" / "khung1_embeddings.npz"

# Config — set SKIP_VOYAGE=True if you hit Voyage 3 RPM billing limits
SKIP_VOYAGE = os.getenv("SKIP_VOYAGE", "0") == "1"
RUN_DRAFTS = os.getenv("RUN_DRAFTS", "0") == "1"
HYBRID_K = 20
VOYAGE_K = 5
SAG_EXTRA = 10
MIN_SIM = 0.6
VOYAGE_SLEEP_S = 22  # stay under ~3 RPM when Voyage is enabled

settings = get_settings()
print("ROOT", ROOT)
print("gold", GOLD_PATH.is_file(), "| corpus", RAW_JSON.is_file())
print("SKIP_VOYAGE", SKIP_VOYAGE, "| RUN_DRAFTS", RUN_DRAFTS)
print("voyage_configured", settings.voyage_configured, "| qwen_configured", settings.qwen_configured)
print("Khung1 docs", len(KHUNG1_DOC_IDS))

ROOT /Users/davidmai/Documents/Work/SAG
gold True | corpus True
SKIP_VOYAGE False | RUN_DRAFTS False
voyage_configured True | qwen_configured True
Khung1 docs 14


## 5. Load gold + corpus index

In [2]:
gold = load_gold(GOLD_PATH)
print(f"Loaded {len(gold)} gold items")
for g in gold:
    print(f"  - {g.id} [{g.criterion}]: {g.query[:70]}…")

print("\nIngesting Khung 1…")
chunks = flatten_chunks(ingest_corpus(RAW_JSON, doc_ids=KHUNG1_DOC_IDS))
print(f"chunks={len(chunks)}")
print("Embedding (uses cache if ids match)…")
vectors = embed_chunks(chunks, cache_path=CACHE_NPZ)
index = build_index(chunks, vectors=vectors, min_sim=MIN_SIM)
print(
    f"SAG index: events={len(index.events_by_id)} "
    f"entities={len(index.events_by_entity)}"
)

Loaded 5 gold items
  - q01_dieu40_orphan [orphan_repair]: Cuối kỳ kế toán năm thì đơn vị kế toán phải làm nghĩa vụ gì theo Điều …
  - q02_aml_tctd [cross_law]: Tổ chức tín dụng phải làm gì để phòng chống rửa tiền?…
  - q03_ky_ke_toan [lexical_anchor]: Kỳ kế toán năm được quy định thế nào?…
  - q04_nhan_hieu [definition_anchor]: Nhãn hiệu là gì theo Luật Sở hữu trí tuệ?…
  - q05_pcrt_nguyen_tac [parent_repair]: Các nguyên tắc phòng chống rửa tiền theo luật là gì?…

Ingesting Khung 1…
chunks=8644
Embedding (uses cache if ids match)…
SAG index: events=8469 entities=5508


## 6. Run RAG vs SAG retrieval and score

In [3]:
def retrieve_arms(query: str):
    hybrid = search_hybrid(query, chunks, k=HYBRID_K, vectors=vectors)
    hybrid_chunks = [h.chunk for h in hybrid]
    if SKIP_VOYAGE or not settings.voyage_configured:
        seeds = hybrid_chunks[:VOYAGE_K]
        mode = "hybrid-as-seeds"
    else:
        time.sleep(VOYAGE_SLEEP_S)
        seeds = [h.chunk for h in rerank(query, hybrid_chunks, top_k=VOYAGE_K)]
        mode = "voyage"
    rag = list(seeds)
    sag = expand(seeds, index, max_extra=SAG_EXTRA, hops=1, min_sim=MIN_SIM)
    return mode, rag, sag


rows = []
detail_rows = []

for g in gold:
    mode, rag_ev, sag_ev = retrieve_arms(g.query)
    rag_s = score_retrieval(rag_ev, g)
    sag_s = score_retrieval(sag_ev, g)

    rag_draft = sag_draft = None
    rag_a = sag_a = None
    if RUN_DRAFTS and settings.qwen_configured:
        seed_ids = {c.chunk_id for c in rag_ev}
        rag_draft = generate_draft(g.query, rag_ev, seed_ids=seed_ids)
        sag_draft = generate_draft(g.query, sag_ev, seed_ids=seed_ids)
        rag_a = score_answer(rag_draft.answer, g, abstained=rag_draft.abstained)
        sag_a = score_answer(sag_draft.answer, g, abstained=sag_draft.abstained)

    for arm, ev, rs, ans in [
        ("RAG", rag_ev, rag_s, rag_a),
        ("SAG", sag_ev, sag_s, sag_a),
    ]:
        rows.append(
            {
                "id": g.id,
                "criterion": g.criterion,
                "arm": arm,
                "seed_mode": mode,
                "K": rs.k,
                "doc_recall": rs.doc_recall,
                "prov_recall": rs.provision_recall,
                "precision@K": rs.precision_at_k,
                "f1_prov": f1(rs.precision_at_k, rs.provision_recall),
                "phrase_recall": (ans.phrase_recall if ans else None),
                "abstained": (ans.abstained if ans else None),
            }
        )
        detail_rows.append(
            {
                "id": g.id,
                "arm": arm,
                "missing_docs": rs.missing_docs,
                "missing_provisions": rs.missing_provisions,
                "missing_phrases": (ans.missing_phrases if ans else None),
            }
        )

df = pd.DataFrame(rows)
df

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,id,criterion,arm,seed_mode,K,doc_recall,prov_recall,precision@K,f1_prov,phrase_recall,abstained
0,q01_dieu40_orphan,orphan_repair,RAG,voyage,5,1.0,1.0,0.200000,0.333333,None,None
1,q01_dieu40_orphan,orphan_repair,SAG,voyage,15,1.0,1.0,0.200000,0.333333,None,None
2,q02_aml_tctd,cross_law,RAG,voyage,5,1.0,1.0,0.400000,0.571429,None,None
3,q02_aml_tctd,cross_law,SAG,voyage,15,1.0,1.0,0.466667,0.636364,None,None
4,q03_ky_ke_toan,lexical_anchor,RAG,voyage,5,1.0,1.0,0.600000,0.750000,None,None
5,q03_ky_ke_toan,lexical_anchor,SAG,voyage,15,1.0,1.0,0.400000,0.571429,None,None
6,q04_nhan_hieu,definition_anchor,RAG,voyage,5,1.0,0.0,0.000000,0.000000,None,None
7,q04_nhan_hieu,definition_anchor,SAG,voyage,15,1.0,0.0,0.000000,0.000000,None,None
8,q05_pcrt_nguyen_tac,parent_repair,RAG,voyage,5,1.0,1.0,0.600000,0.750000,None,None
9,q05_pcrt_nguyen_tac,parent_repair,SAG,voyage,15,1.0,1.0,0.200000,0.333333,None,None


## 7. Leaderboard (mean over gold)

In [4]:
summary = (
    df.groupby("arm")[["doc_recall", "prov_recall", "precision@K", "f1_prov"]]
    .mean()
    .reset_index()
)
if df["phrase_recall"].notna().any():
    summary = summary.merge(
        df.groupby("arm")["phrase_recall"].mean().reset_index(),
        on="arm",
    )

print("=== Mean scores (higher is better, except interpret Precision carefully) ===")
display(summary)

rag_prov = float(summary.loc[summary["arm"] == "RAG", "prov_recall"].iloc[0])
sag_prov = float(summary.loc[summary["arm"] == "SAG", "prov_recall"].iloc[0])
print(f"\nΔ ProvRecall (SAG − RAG) = {sag_prov - rag_prov:+.3f}")

print("\n=== Per-question misses ===")
pd.DataFrame(detail_rows)

=== Mean scores (higher is better, except interpret Precision carefully) ===


,arm,doc_recall,prov_recall,precision@K,f1_prov
0,RAG,1.0,0.8,0.360000,0.480952
1,SAG,1.0,0.8,0.253333,0.374892



Δ ProvRecall (SAG − RAG) = +0.000

=== Per-question misses ===


,id,arm,missing_docs,missing_provisions,missing_phrases
0,q01_dieu40_orphan,RAG,[],[],None
1,q01_dieu40_orphan,SAG,[],[],None
2,q02_aml_tctd,RAG,[],[],None
3,q02_aml_tctd,SAG,[],[],None
4,q03_ky_ke_toan,RAG,[],[],None
5,q03_ky_ke_toan,SAG,[],[],None
6,q04_nhan_hieu,RAG,[],[law-2005-luat-so-huu-tri-tue | Điều 4 | Khoản...,None
7,q04_nhan_hieu,SAG,[],[law-2005-luat-so-huu-tri-tue | Điều 4 | Khoản...,None
8,q05_pcrt_nguyen_tac,RAG,[],[],None
9,q05_pcrt_nguyen_tac,SAG,[],[],None


## 8. How to interpret results

1. **ProvRecall↑ with SAG, Precision↓** — expected when expansion adds parents/siblings; still a win if answers need those parents (see q01 Điều 40).  
2. **DocRecall flat, ProvRecall↑** — same statute, better internal coverage (orphan/parent repair).  
3. **Both arms fail DocRecall** — routing/corpus problem (wrong law in dump, or query underspecified), not fixed by SAG alone.  
4. **ProvRecall↑ but PhraseRecall flat** — retrieval fixed, draft selection/prompt still drops repairs (we already hit this once).  
5. **Control q03** — if SAG does not hurt Doc/Prov recall here, expansion is not breaking easy lexical cases.

### Re-run tips
```bash
# avoid Voyage rate limits
SKIP_VOYAGE=1 jupyter notebook notebooks/eval_khung1_rag_vs_sag.ipynb

# also score drafts (needs QWEN_API_KEY)
RUN_DRAFTS=1 SKIP_VOYAGE=1 jupyter notebook notebooks/eval_khung1_rag_vs_sag.ipynb
```